# Load Data and Extract Audio

In [27]:
import subprocess
import os

DataFolder = 'Data/TestData1'

mov_path = None
wav_path = None
py_path = None
VIDEO_CLAP_FRAME = None

def extract_audio_from_mov(mov_path, wav_path):
    """
    Helper command to strictly extract .wav standard acoustics natively from Apple .mov containers using OSX afconvert.
    """
    subprocess.run(['afconvert', '-f', 'WAVE', '-d', 'LEI16', mov_path, wav_path])
    print(f"Extraction complete: Saved standalone wave audio to {wav_path}")

for filename in os.listdir(DataFolder):
    if filename.endswith('.mov'):
        mov_path = os.path.join(DataFolder, filename)
        wav_path = os.path.join(DataFolder, filename.replace('.mov', '.wav'))
        py_path = os.path.join(DataFolder, filename.replace('.mov', '.blender.py'))
        frame = os.path.join(DataFolder, "frame.txt")
        VIDEO_CLAP_FRAME = int(open(frame).read().strip())

        extract_audio_from_mov(mov_path, wav_path)

Extraction complete: Saved standalone wave audio to Data/TestData1/ar-captured-1779403063.wav


In [28]:
# %%
import numpy as np
import pandas as pd
import scipy.signal as signal
from scipy.io import wavfile
from scipy.spatial.transform import Rotation, Slerp
import cv2
import plotly.graph_objects as go
import ast
import re
from sklearn.linear_model import RANSACRegressor

# =====================================================================
# 1. PARSING & SYNCHRONIZATION 
# =====================================================================

def parse_blender_tracking_log(py_path, fps=60.0):
    with open(py_path, 'r') as f:
        content = f.read()
    fps_match = re.search(r'bpy\.context\.scene\.render\.fps\s*=\s*(\d+)', content)
    fps = float(fps_match.group(1)) if fps_match else fps
    
    frames = ast.literal_eval(re.search(r'movement_keyframes\s*=\s*(\[.*?\])', content, re.DOTALL).group(1))
    locations = ast.literal_eval(re.search(r'locations\s*=\s*(\[.*?\])', content, re.DOTALL).group(1))
    rotations = ast.literal_eval(re.search(r'rotations\s*=\s*(\[.*?\])', content, re.DOTALL).group(1))
    
    timestamps, translations, rotation_matrices = [], [], []
    for frame, loc, rot in zip(frames, locations, rotations):
        timestamps.append(frame / fps)
        translations.append(loc)
        # Assuming Blender ZXY Euler
        r = Rotation.from_euler('ZXY', [rot[2], rot[0], rot[1]], degrees=False)
        rotation_matrices.append(r.as_matrix())
        
    return pd.DataFrame({'timestamp': timestamps, 'translation': translations, 'rotation_matrix': rotation_matrices})

def parse_blender_intrinsics(py_path):
    with open(py_path, 'r') as f:
        content = f.read()
    width = float(re.search(r'resolution_x\s*=\s*([\d.]+)', content).group(1))
    height = float(re.search(r'resolution_y\s*=\s*([\d.]+)', content).group(1))
    lens = float(re.search(r'lens\s*=\s*([\d.]+)', content).group(1))
    sensor_width = float(re.search(r'sensor_width\s*=\s*([\d.]+)', content).group(1))
    
    fx = (lens / sensor_width) * width
    # Assuming square pixels, fy = fx
    return np.array([[fx, 0, width/2.0], [0, fx, height/2.0], [0, 0, 1]]), int(width), int(height)

# =====================================================================
# 2. SIGNAL PROCESSING (WITH SNR CONSTRAINT)
# =====================================================================

def compute_acoustic_distances(audio_sig, trajectory_df, sample_rate, reference_chirp, snr_threshold=3.0):
    SPEED_OF_SOUND = 343.0  
    if audio_sig.ndim > 1: audio_sig = audio_sig[:, 0]
    timestamps = trajectory_df['timestamp'].to_numpy()
    distances = np.full(len(timestamps), np.nan)
    window_samples = int(0.100 * sample_rate)
    
    center_indices = (timestamps * sample_rate).astype(int)
    for i, idx_center in enumerate(center_indices):
        if idx_center + window_samples > len(audio_sig): continue
        audio_window = audio_sig[idx_center : idx_center + window_samples]
        if len(audio_window) < len(reference_chirp): continue
            
        xcorr = signal.correlate(audio_window, reference_chirp, mode='valid')
        
        # SNR Calculation Constraint
        noise_floor = np.sqrt(np.mean(audio_window**2)) + 1e-6 
        peaks, properties = signal.find_peaks(xcorr, height=noise_floor * snr_threshold, distance=int(0.001 * sample_rate))
        
        if len(peaks) >= 2:
            delay = (peaks[1] - peaks[0]) / sample_rate
            dist = (SPEED_OF_SOUND * delay) / 2.0
            if dist <= 1.2: 
                distances[i] = dist
    return distances

# =====================================================================
# 3. 3D HOUGH SPACE MAPPING
# =====================================================================

def build_hough_accumulator(pose_df, acoustic_distances, voxel_size=0.04, margin=1.5):
    trajectory = np.array(pose_df['translation'].tolist())
    rotations = np.array(pose_df['rotation_matrix'].tolist())
    min_bounds = np.min(trajectory, axis=0) - margin
    max_bounds = np.max(trajectory, axis=0) + margin
    
    grid_dims = np.ceil((max_bounds - min_bounds) / voxel_size).astype(int)
    voxel_grid = np.zeros(grid_dims, dtype=np.int32)
    
    x_bins = min_bounds[0] + np.arange(grid_dims[0]) * voxel_size
    y_bins = min_bounds[1] + np.arange(grid_dims[1]) * voxel_size
    z_bins = min_bounds[2] + np.arange(grid_dims[2]) * voxel_size
    
    for i, (C_i, d_i) in enumerate(zip(trajectory, acoustic_distances)):
        if np.isnan(d_i) or d_i <= 0: continue
        
        # Local 3D Sub-Grid Bounding Box (Crucial Constraint)
        idx_min = np.maximum(0, np.floor((C_i - (d_i + voxel_size) - min_bounds) / voxel_size)).astype(int)
        idx_max = np.minimum(grid_dims, np.ceil((C_i + (d_i + voxel_size) - min_bounds) / voxel_size)).astype(int)
        
        ix, iy, iz = np.arange(idx_min[0], idx_max[0]), np.arange(idx_min[1], idx_max[1]), np.arange(idx_min[2], idx_max[2])
        if len(ix)==0 or len(iy)==0 or len(iz)==0: continue

        X_s, Y_s, Z_s = np.meshgrid(x_bins[ix], y_bins[iy], z_bins[iz], indexing='ij')
        
        dist_to_cam = np.sqrt((X_s - C_i[0])**2 + (Y_s - C_i[1])**2 + (Z_s - C_i[2])**2)
        forward_world = -rotations[i][:, 2] 
        dot_prod = ((X_s - C_i[0])*forward_world[0] + (Y_s - C_i[1])*forward_world[1] + (Z_s - C_i[2])*forward_world[2]) / (dist_to_cam + 1e-6)
        
        # Increment voxels on the sphere shell within FOV
        voxel_grid[idx_min[0]:idx_max[0], idx_min[1]:idx_max[1], idx_min[2]:idx_max[2]] += (np.abs(dist_to_cam - d_i) < voxel_size) & (dot_prod >= 0.707)
        
    return voxel_grid, min_bounds, voxel_size

# =====================================================================
# 4. RECTANGULAR GEOMETRY EXTRUSION & PROJECTION
# =====================================================================

def generate_balanced_room_mesh(voxel_points, trajectory_df, residual_threshold=0.08, resolution=0.03):
    """
    Uses Spatial RANSAC purely as a noise filter to extract the dense wall cores.
    Then applies a global Minimum Area Rectangle to the clean points to find a 
    perfectly balanced, orthogonal 4-wall room without 'first-wall bias'.
    """
    points_2d = np.float32(voxel_points[:, [0, 2]])
    
    # --- 1. RANSAC DENOISER ---
    clean_wall_points = []
    pts_pool = points_2d.copy()
    
    for _ in range(10): # Hunt for all possible dense clusters
        if len(pts_pool) < 20: break
            
        best_inliers = []
        for _ in range(500):
            idx = np.random.choice(len(pts_pool), 2, replace=False)
            p1, p2 = pts_pool[idx]
            dx, dz = p2[0] - p1[0], p2[1] - p1[1]
            length = np.hypot(dx, dz)
            if length < 0.01: continue
            
            A, B = -dz / length, dx / length
            C = -(A * p1[0] + B * p1[1])
            
            dist = np.abs(A * pts_pool[:, 0] + B * pts_pool[:, 1] + C)
            inliers = np.where(dist < residual_threshold)[0]
            
            if len(inliers) > len(best_inliers):
                best_inliers = inliers
                
        if len(best_inliers) >= 20:
            clean_wall_points.append(pts_pool[best_inliers])
            pts_pool = np.delete(pts_pool, best_inliers, axis=0)
        else:
            break # No more dense lines found
            
    # Fallback if the data is completely sparse
    if not clean_wall_points:
        combined_clean_points = points_2d
    else:
        combined_clean_points = np.vstack(clean_wall_points)

    # --- 2. GLOBAL BALANCED ORTHOGONAL FIT ---
    # Finds the optimal rectangle balancing all clean wall points
    rect = cv2.minAreaRect(combined_clean_points)
    box = cv2.boxPoints(rect)
    
    # --- 3. EXTRUDE MESH ---
    y_min = trajectory_df['translation'].apply(lambda x: x[1]).min() - 0.2
    y_max = trajectory_df['translation'].apply(lambda x: x[1]).max() + 1.5
    height_grid = np.arange(y_min, y_max, resolution)
    
    dense_points = []
    for i in range(4):
        p1 = box[i]
        p2 = box[(i + 1) % 4]
        
        dist = np.linalg.norm(p2 - p1)
        num_steps = max(2, int(dist / resolution))
        
        x_line = np.linspace(p1[0], p2[0], num_steps)
        z_line = np.linspace(p1[1], p2[1], num_steps)
        
        for h in height_grid:
            dense_points.append(np.column_stack([x_line, np.full_like(x_line, h), z_line]))
            
    return np.vstack(dense_points) if dense_points else np.empty((0, 3))

def project_textures_to_mesh(mesh_points, video_path, pose_df, K, video_clap_frame, max_keyframes=45):
    cap = cv2.VideoCapture(video_path)
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    
    point_colors = np.ones((len(mesh_points), 3)) * 0.15 # Fallback dark gray
    best_depth = np.full(len(mesh_points), np.inf) # Z-Buffer for Winner-Takes-All
    
    times = pose_df['timestamp'].values
    t_min, t_max = times.min(), times.max()
    
    # 1. Collect all valid frames
    candidates, frame_idx = [], 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        t_query = (frame_idx - video_clap_frame) / video_fps
        if t_min <= t_query <= t_max:
            if frame_idx % 10 == 0:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                candidates.append((frame_idx, np.var(gray), t_query))
        frame_idx += 1
        
    # 2. CRITICAL FIX: Temporal NMS Keyframe Selection
    candidates.sort(key=lambda x: x[1], reverse=True)
    keyframes = []
    
    for cand in candidates:
        # Enforce a strict 1-second minimum gap between selected frames
        # This creates massive, solid patches and eliminates jagged micro-seams
        is_distinct = True
        for kf in keyframes:
            if abs(cand[2] - kf[2]) < 1: 
                is_distinct = False
                break
                
        if is_distinct:
            keyframes.append(cand)
            
        if len(keyframes) >= max_keyframes:
            break
            
    translations = np.array(pose_df['translation'].tolist())
    rotations = np.array(pose_df['rotation_matrix'].tolist())
    
    # 3. Strict Winner-Takes-All Projection (No Blending = No Ghosting)
    for f_idx, variance, t_query in keyframes:
        idx = np.clip(np.searchsorted(times, t_query) - 1, 0, len(times) - 2)
        weight_t = (t_query - times[idx]) / (times[idx+1] - times[idx])
        
        t_w = (1 - weight_t) * translations[idx] + weight_t * translations[idx+1]
        slerp = Slerp(times[idx:idx+2], Rotation.from_matrix(rotations[idx:idx+2]))
        R_cw = slerp([t_query]).as_matrix()[0]
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
        ret, frame = cap.read()
        if not ret: continue
            
        X_centered = mesh_points - t_w
        X_cam = X_centered @ R_cw 
        
        X_opencv = np.zeros_like(X_cam)
        X_opencv[:, 0] =  X_cam[:, 0]  
        X_opencv[:, 1] = -X_cam[:, 1]  
        X_opencv[:, 2] = -X_cam[:, 2]  
        
        valid_depth = X_opencv[:, 2] > 0.1
        valid_idx = np.where(valid_depth)[0]
        if len(valid_idx) == 0: continue
            
        X_v = X_opencv[valid_idx]
        u = (X_v[:, 0] * K[0,0] / X_v[:, 2]) + K[0,2]
        v = (X_v[:, 1] * K[1,1] / X_v[:, 2]) + K[1,2]
        
        in_frame = (u >= 0) & (u < width - 1) & (v >= 0) & (v < height - 1)
        in_frame_idx = valid_idx[in_frame]
        if len(in_frame_idx) == 0: continue
            
        current_depths = X_v[in_frame, 2]
        better_view_mask = current_depths < best_depth[in_frame_idx]
        
        if not np.any(better_view_mask): continue
            
        update_idx = in_frame_idx[better_view_mask]
        u_win = u[in_frame][better_view_mask]
        v_win = v[in_frame][better_view_mask]
        
        u0, v0 = np.floor(u_win).astype(int), np.floor(v_win).astype(int)
        u1, v1 = u0 + 1, v0 + 1
        wu, wv = u_win - u0, v_win - v0
        
        sampled = (
            frame[v0, u0] * (1 - wu)[:, None] * (1 - wv)[:, None] +
            frame[v0, u1] * wu[:, None] * (1 - wv)[:, None] +
            frame[v1, u0] * (1 - wu)[:, None] * wv[:, None] +
            frame[v1, u1] * wu[:, None] * wv[:, None]
        )
        
        # Pure overwrite. 
        point_colors[update_idx] = sampled[:, ::-1] / 255.0
        best_depth[update_idx] = current_depths[better_view_mask]
        
    cap.release()
    point_colors = np.clip(point_colors, 0.0, 1.0)
    
    colors_uint8 = (point_colors * 255).astype(np.uint8)
    plotly_colors = [f'rgb({c[0]}, {c[1]}, {c[2]})' for c in colors_uint8]
    
    return plotly_colors


# =====================================================================
# 5. VISUALIZATION
# =====================================================================

def display_reconstruction_results(mesh_points, mesh_colors, hough_points, trajectory_df):
    trajectory = np.array(trajectory_df['translation'].tolist())
    
    # ---------------------------------------------------------
    # PLOT 1: THE BEAUTY PASS (Clean Textured Room)
    # ---------------------------------------------------------
    trace_clean_mesh = go.Scatter3d(
        x=mesh_points[:, 0], y=mesh_points[:, 1], z=mesh_points[:, 2],
        mode='markers',
        marker=dict(size=4, color=mesh_colors, opacity=1.0),
        name='Textured Room Mesh'
    )
    
    layout_clean = go.Layout(
        title="Final Output: Clean Textured Room Reconstruction",
        scene=dict(
            xaxis=dict(title='X (m)', backgroundcolor='black', gridcolor='#333333'),
            yaxis=dict(title='Y (m)', backgroundcolor='black', gridcolor='#333333'),
            zaxis=dict(title='Z (m)', backgroundcolor='black', gridcolor='#333333'),
            bgcolor='black', aspectmode='data'
        ),
        paper_bgcolor='black', font=dict(color='white'), margin=dict(l=0, r=0, t=40, b=0)
    )
    go.Figure(data=[trace_clean_mesh], layout=layout_clean).show()

    # ---------------------------------------------------------
    # PLOT 2: THE DIAGNOSTIC PASS (Geometry vs Raw Data)
    # ---------------------------------------------------------
    traces_diagnostic = [
        go.Scatter3d(
            x=mesh_points[:, 0], y=mesh_points[:, 1], z=mesh_points[:, 2],
            mode='markers',
            marker=dict(size=4, color=mesh_colors, opacity=0.9), # Slightly transparent
            name='Textured Room Mesh'
        ),
        go.Scatter3d(
            x=hough_points[:, 0], y=hough_points[:, 1], z=hough_points[:, 2],
            mode='markers',
            marker=dict(size=2, color='rgba(0, 255, 255, 0.3)'), # Ghosted cyan
            name='Raw Acoustic Voxels'
        ),
        go.Scatter3d(
            x=trajectory[:, 0], y=trajectory[:, 1], z=trajectory[:, 2],
            mode='lines+markers',
            line=dict(color='yellow', width=4),
            marker=dict(size=2, color='white'),
            name='Camera Path'
        )
    ]
    
    layout_diagnostic = go.Layout(
        title="Diagnostic View: Geometry vs. Raw Acoustic Data",
        scene=dict(
            xaxis=dict(title='X (m)', backgroundcolor='black', gridcolor='#333333'),
            yaxis=dict(title='Y (m)', backgroundcolor='black', gridcolor='#333333'),
            zaxis=dict(title='Z (m)', backgroundcolor='black', gridcolor='#333333'),
            bgcolor='black', aspectmode='data'
        ),
        paper_bgcolor='black', font=dict(color='white'), margin=dict(l=0, r=0, t=40, b=0)
    )
    go.Figure(data=traces_diagnostic, layout=layout_diagnostic).show()

# =====================================================================
# MASTER EXECUTION 
# =====================================================================
if __name__ == "__main__":
    print("Initializing Active Room Reconstruction Engine...")
    
    VIDEO_FPS = 60.0

    print("Parsing telemetry...")
    trajectory_df = parse_blender_tracking_log(py_path, fps=VIDEO_FPS)
    K, video_width, video_height = parse_blender_intrinsics(py_path)
    sr, audio_sig = wavfile.read(wav_path)

    # Reference signal
    t_chirp = np.linspace(0, 0.010, int(44100 * 0.010), endpoint=False)
    tapered_chirp = np.sin(2 * np.pi * (18000 + (22000 - 18000) * t_chirp / (2 * 0.010))) * signal.windows.tukey(len(t_chirp), alpha=0.1)

    # Align Timelines
    trajectory_df['timestamp'] = trajectory_df['timestamp'] - (VIDEO_CLAP_FRAME / VIDEO_FPS)

    print("Extracting acoustic ToF distances (SNR filtered)...")
    acoustic_distances = compute_acoustic_distances(audio_sig, trajectory_df, sr, tapered_chirp)
    
    print("Accumulating 3D Hough Space...")
    voxel_grid, bounds, v_scale = build_hough_accumulator(trajectory_df, acoustic_distances, voxel_size=0.04)

    print("Extracting high-confidence acoustic voxels...")
    # Extract high-confidence raw voxel points (The Hough Map)
    voxel_indices = np.argwhere(voxel_grid >= np.percentile(voxel_grid[voxel_grid > 0], 95))
    hough_points = bounds + (voxel_indices * v_scale)

    # CRITICAL FIX: Calling the new RANSAC function instead of the old Rectangle function
    print("Calculating Iterative RANSAC Planes...")
    ideal_room_mesh = generate_balanced_room_mesh(hough_points, trajectory_df, residual_threshold=0.08, resolution=0.03)

    print(f"Projecting textures onto {len(ideal_room_mesh)} mesh points...")
    mesh_colors = project_textures_to_mesh(ideal_room_mesh, mov_path, trajectory_df, K, VIDEO_CLAP_FRAME)

    print("Launching final geometry scenes...")
    display_reconstruction_results(ideal_room_mesh, mesh_colors, hough_points, trajectory_df)
    print("Execution complete!")

Initializing Active Room Reconstruction Engine...
Parsing telemetry...
Extracting acoustic ToF distances (SNR filtered)...


/var/folders/cc/298066sj03bgxmkbtmzgvb7h0000gn/T/ipykernel_32283/614190236.py:380: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, audio_sig = wavfile.read(wav_path)


Accumulating 3D Hough Space...
Extracting high-confidence acoustic voxels...
Calculating Iterative RANSAC Planes...
Projecting textures onto 34980 mesh points...
Launching final geometry scenes...


Execution complete!
